# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step exploratory template for loading and analyzing a Croissant dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata object (not as a dict)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}")
print("\nCite As: %s" % getattr(metadata, 'citeAs', ''))
print("License:", metadata.license)
print("Date Published:", metadata.datePublished)
print("Authors/IDs:", [a["@id"] if isinstance(a, dict) and "@id" in a else a for a in getattr(metadata, 'author', [])])
print("Distribution IDs:", [d["@id"] if isinstance(d, dict) and "@id" in d else d for d in getattr(metadata, 'distribution', [])])

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities are referenced by their `@id` fields.

Let's enumerate record sets, fields, and columns. (Record sets contain the dataset's tables.)

In [ ]:
# List record sets and their fields/columns using @id
record_sets_info = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []

record_set_ids = []
fields_by_record_set = {}

for rs in record_sets_info:
    if isinstance(rs, dict) and '@id' in rs:
        rs_id = rs['@id']
    else:
        rs_id = rs
    record_set_ids.append(rs_id)

    # Attempt to fetch fields
    if hasattr(dataset.metadata, 'field'):
        fields = [f['@id'] for f in dataset.metadata.field if isinstance(f, dict) and f.get('recordSet') == rs_id]
        fields_by_record_set[rs_id] = fields
    else:
        fields_by_record_set[rs_id] = []

print("Record Set IDs:", record_set_ids)

# For each record set, print fields
for rs_id in record_set_ids:
    print(f"\nRecord Set @id: {rs_id}")
    if fields_by_record_set[rs_id]:
        print("Field @ids:", fields_by_record_set[rs_id])
    else:
        print("No fields found (fields may be embedded in columns)")

# Show sample records from the first record set
if record_set_ids:
    for i, rec in enumerate(dataset.records(record_set=record_set_ids[0])):
        print(f"Record {i}:", rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Note: Entities are referenced by their `@id`.

We'll load the records from each record set defined in the schema.

In [ ]:
# Extract all record sets into DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    # Load records
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded {len(df)} records for record set {record_set_id}")
        print("Columns:", df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load records for record set {record_set_id}: {e}")

# For demonstration, pick the first record set
main_rs_id = record_set_ids[0] if record_set_ids else None
if main_rs_id:
    main_df = dataframes[main_rs_id]
    print(f"\nFirst record set ({main_rs_id}) columns:")
    print(main_df.columns.tolist())
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Let's process the data: filter records, normalize numeric fields, remove outliers, and group by a categorical column.

All references use the `@id` for fields/columns. Adjust `numeric_field_id` and `group_field_id` as needed based on the available columns.

In [ ]:
# Identify some numeric and group fields by their @id (replace as per actual columns)
if main_rs_id:
    main_columns = dataframes[main_rs_id].columns
    pprint.pprint(list(main_columns))

# Example: Let's pick 'GAD-7_total_score' and 'gender' as fields
# Suppose their @ids are 'GAD7_total_score' and 'gender' (replace with actual @id as needed)
numeric_field_id = 'GAD7_total_score' if 'GAD7_total_score' in main_columns else (main_columns[0] if len(main_columns)>0 else None)
group_field_id = 'gender' if 'gender' in main_columns else (main_columns[1] if len(main_columns)>1 else None)

# Filter for scores above a threshold
if numeric_field_id and main_rs_id:
    threshold = 10
    filtered_df = dataframes[main_rs_id][dataframes[main_rs_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field and calculate mean
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        )
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df)

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

For example, plot the normalized score distribution and compare means across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histograms of normalized scores if available
if main_rs_id and numeric_field_id and f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=20)
    plt.title(f"Distribution of Normalized {numeric_field_id} Scores")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.show()

# Bar plot: Mean score by group
if group_field_id and numeric_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(6, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
Using the `mlcroissant` library, we loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya. We accessed entities by their `@id`, filtered and normalized numeric fields, and visualized group-level differences.

This Croissant-standard dataset is well-structured and ready for downstream analysis or machine learning tasks, demonstrating best practices for AI-ready, FAIR-compliant data infrastructure in Africa.